In [19]:
import pandas as pd
import numpy as np

# Setting seed so you get the exact same data every time
np.random.seed(42)
n_samples = 500

data = {
    'customer_age': np.random.choice([18, 22, 35, 45, 60, np.nan], size=n_samples, p=[0.2, 0.2, 0.2, 0.2, 0.1, 0.1]),
    'device_type': np.random.choice(['Mobile', 'Desktop', 'Tablet', np.nan], size=n_samples, p=[0.6, 0.3, 0.05, 0.05]),
    'product_category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Beauty'], size=n_samples),
    'item_price': np.round(np.random.uniform(10, 500, size=n_samples), 2),
    'discount_percentage': np.random.choice([0, 10, 20, 50, np.nan], size=n_samples, p=[0.4, 0.3, 0.15, 0.1, 0.05]),
    'is_first_time_buyer': np.random.choice(['Yes', 'No'], size=n_samples, p=[0.3, 0.7]),
    'returned': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]) # 1 = Returned, 0 = Kept
}

df = pd.DataFrame(data)
df.to_csv('ecommerce_returns.csv', index=False)
print("Dataset successfully created and saved as 'ecommerce_returns.csv'!")

Dataset successfully created and saved as 'ecommerce_returns.csv'!


In [23]:
df.sample(10)

,customer_age,device_type,product_category,item_price,discount_percentage,is_first_time_buyer,returned
327,45.0,Mobile,Home,324.57,10.0,No,0
326,35.0,Mobile,Clothing,21.58,0.0,No,1
197,60.0,Mobile,Beauty,420.68,0.0,Yes,0
0,22.0,Desktop,Home,400.30,10.0,Yes,0
120,60.0,Mobile,Beauty,485.98,20.0,No,1
159,45.0,Mobile,Electronics,435.45,0.0,No,0
303,45.0,Desktop,Electronics,443.26,50.0,No,0
282,22.0,Mobile,Beauty,360.91,10.0,Yes,0
140,NaN,Mobile,Beauty,125.66,0.0,No,0
245,NaN,Desktop,Clothing,337.42,10.0,No,1


In [21]:
df.shape

(500, 7)

In [22]:
df.dtypes

customer_age           float64
device_type             object
product_category        object
item_price             float64
discount_percentage    float64
is_first_time_buyer     object
returned                 int64
dtype: object

In [24]:
df.isnull().sum()

customer_age           55
device_type             0
product_category        0
item_price              0
discount_percentage    25
is_first_time_buyer     0
returned                0
dtype: int64

In [51]:
df.sample()

,customer_age,device_type,product_category,item_price,discount_percentage,is_first_time_buyer,returned
435,45.0,Mobile,Beauty,432.39,50.0,Yes,0


In [63]:
df['device_type'].unique()

array(['Desktop', 'Mobile', 'Tablet', 'nan'], dtype=object)

In [73]:
df['product_category'].unique()

array(['Home', 'Beauty', 'Clothing', 'Electronics'], dtype=object)

In [74]:
df['is_first_time_buyer'].unique()

array(['Yes', 'No'], dtype=object)

In [33]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [46]:
# 1.separating features and target
x = df.drop(columns=['returned'])
y = df['returned']

In [43]:
# t2.train test split
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [47]:
df.sample()

,customer_age,device_type,product_category,item_price,discount_percentage,is_first_time_buyer,returned
455,35.0,Mobile,Electronics,61.78,50.0,No,1


In [48]:
# 3.define the columns
num_cols = ['customer_age','item_price','discount_percentage']
cat_cols = ['device_type','product_category','is_first_time_buyer']

In [49]:
# 4.Creating a pipeline for numerical features

num_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

In [68]:
# 5.Creating a pipeline for categorica features
cat_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'))
])

In [70]:
# 6. Combining into ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num',num_pipeline,num_cols),
    ('cat',cat_pipeline,cat_cols)
])

In [72]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)


print("Original X_train shape:", X_train.shape)
print("Transformed X_train shape:", X_train_transformed.shape)

Original X_train shape: (400, 6)
Transformed X_train shape: (400, 10)


In [77]:
print(preprocessor.get_feature_names_out())

['num__customer_age' 'num__item_price' 'num__discount_percentage'
 'cat__device_type_Mobile' 'cat__device_type_Tablet'
 'cat__device_type_nan' 'cat__product_category_Clothing'
 'cat__product_category_Electronics' 'cat__product_category_Home'
 'cat__is_first_time_buyer_Yes']


In [78]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Combine Preprocessor + Model into a single master Pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# 2. Train the ENTIRE pipeline on raw X_train, y_train in ONE line!
full_pipeline.fit(X_train, y_train)

# 3. Predict directly on RAW X_test!
y_pred = full_pipeline.predict(X_test)

# 4. Evaluate performance
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.78

Classification Report:
               precision    recall  f1-score   support

           0       0.78      1.00      0.88        78
           1       0.00      0.00      0.00        22

    accuracy                           0.78       100
   macro avg       0.39      0.50      0.44       100
weighted avg       0.61      0.78      0.68       100



C:\Users\Admin\Anaacoda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\Anaacoda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\Anaacoda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [80]:
# Update your pipeline classifier parameter
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced'))
])

# Re-fit and predict
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("NEW Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.57
NEW Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.62      0.69        78
           1       0.23      0.41      0.30        22

    accuracy                           0.57       100
   macro avg       0.51      0.51      0.49       100
weighted avg       0.66      0.57      0.60       100

